# Part 3 · Challenge: you are the station manager
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
With signs, staff and app messages you can steer **which destination people from each gate head for**
(think *"Platform entrance via the South Gate"*). In model terms: **you choose the OD weights.**

### Rules
* Gate demand is fixed.
* The train company requires that **at least 60%** of all travellers are sent to the **Platform X**.
  Both gates are equally busy, so the **average** of your two shares must be **≥ 0.60**.
* **Score = time spent inside (person-hours) + 0.25 h for every person turned away. Lower is better.**
* Post your best valid score on the leaderboard.

**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

## ▶ Step 1: install (run this first, ≈30 s)
This is how you install **any** Python package from GitHub. `pednstream-workshop` is a small
package with this workshop's network and plotting helpers; installing it also installs
**PedNStream** itself. If Colab restarted, run it again.
You can ignore the note *"you may need to restart the kernel"*.

In [ ]:
%pip install -q "pednstream-workshop @ git+https://github.com/WaimenMak/pednstream-workshop.git"

import matplotlib.pyplot as plt
from pednstream_workshop import run, compare, compare_maps
plt.rcParams["figure.dpi"] = 90
print("✅ Ready! Scroll down and continue.")

In [ ]:
#@title 🏆 **Your strategy**: set the values, then press ▶ {display-mode: "form"}
team_name = "my team"  #@param {type:"string"}
share_of_Gate_A_sent_to_Platform = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
share_of_Gate_B_sent_to_Platform = 0.6  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown **Bonus round only**: widen one corridor by 2 m (leave at "none" for the main round)
upgrade = "none"  #@param ["none", "0_2", "1_3", "2_3", "2_4", "3_4", "3_5", "4_6"]

a, b = share_of_Gate_A_sent_to_Platform, share_of_Gate_B_sent_to_Platform
links = {}
if upgrade != "none":
    base_width = run().link(upgrade).width
    links = {upgrade: {"width": base_width + 2}}
res = run(od={"0_5": a, "0_6": 1 - a, "1_5": b, "1_6": 1 - b}, links=links)
k = res.kpis()
score = k["time spent inside (person-hours)"] + 0.25 * k["turned away at gates"]
valid = (a + b) / 2 >= 0.6 - 1e-9

if "attempts" not in globals():
    attempts = []
attempts.append({"team": team_name, "A→X": a, "B→X": b, "upgrade": upgrade,
                 "overall to X": f"{(a + b) / 2:.0%}", "valid": "✅" if valid else "❌", "score": round(float(score), 1)})

print(("✅ Valid strategy" if valid else "❌ BREAKS the 60% rule: this score does not count") +
      f"  (overall {(a + b) / 2:.0%} sent to the Platform)")
print(f"\n   SCORE = {score:.1f}     (time inside {k['time spent inside (person-hours)']:.1f} h, turned away {k['turned away at gates']})\n")
if valid:
    print(f"📋 Leaderboard line:  {team_name} | A→X {a:.2f} | B→X {b:.2f} | upgrade {upgrade} | score {score:.1f}")
res.dashboard(f"{team_name}: A→X {a:.0%}, B→X {b:.0%}"); plt.show()
import pandas as pd
print("Your attempts so far (best first):")
display(pd.DataFrame(attempts).sort_values(["valid", "score"], ascending=[False, True]).reset_index(drop=True))

### Stuck? Hints (open one at a time)
<details><summary>Hint 1</summary>Look at the map of your strategy: which corridors are red, and which OD groups walk through them?</details>
<details><summary>Hint 2</summary>Gate B has only <b>one</b> route to the Platform, and it shares nothing with Gate A's routes.</details>
<details><summary>Hint 3</summary>Remember Exercise 2: the crossing pattern was expensive, the parallel one was cheap.</details>

---
### For coders (optional): search systematically
Instead of guessing, scan every combination and draw a heat-map of the score. Edit and run the cell below.

In [ ]:
import numpy as np
shares = np.round(np.arange(0, 1.01, 0.1), 1)
scores = np.full((len(shares), len(shares)), np.nan)
for i, a in enumerate(shares):
    for j, b in enumerate(shares):
        if (a + b) / 2 < 0.6 - 1e-9:
            continue                       # breaks the 60% rule
        k = run(od={"0_5": a, "0_6": 1 - a, "1_5": b, "1_6": 1 - b}).kpis()
        scores[i, j] = k["time spent inside (person-hours)"] + 0.25 * k["turned away at gates"]

plt.imshow(scores, origin="lower", cmap="viridis_r")
plt.xticks(range(len(shares)), shares); plt.yticks(range(len(shares)), shares)
plt.xlabel("share of Gate B to Platform"); plt.ylabel("share of Gate A to Platform")
plt.colorbar(label="score (lower is better)"); plt.title("Challenge score"); plt.show()
i, j = np.unravel_index(np.nanargmin(scores), scores.shape)
print(f"Best on this grid: A→X {shares[i]}, B→X {shares[j]}, score {scores[i, j]:.1f}")

### Reflection
1. Describe your winning strategy in words a station manager would understand.
2. Why does it work? Name specific corridors.
3. In the bonus round, did one wider corridor help more or less than a smart OD strategy?
4. Name one thing the model leaves out that could make your strategy fail in real life.